IMPORTS

In [4]:
import pandas as pd
import yaml
import litellm

TESTING

In [5]:
test = pd.read_csv("./radiology-reporting-harness/test.csv")
test.head()

,case_id,modality,body_part,study_description,patient_age_band,patient_sex,template_content,dictation
0,00320399-b3ed-447c-be07-a3e3b9af6e2e,CT,Abdomen,CT A/P WO-B,60-64,female,FINDINGS:\nLIVER: Normal in size and attenuati...,Surgically absent gallbladder. Surgical clips ...
1,01b1c9ab-800b-4963-b11d-9d503952e57c,XRAY,Lumbar spine,XR LSP 4V,40-44,female,FINDINGS:\nVERTEBRAE: Normal desnity and align...,Mild lower lumbar facet degenerative changes.\...
2,02aae38a-088b-4cae-bfdc-f0fb38da565d,XRAY,Thoracic spine,XR TSP 2V,50-54,female,FINDINGS:\nVERTEBRAE: Normal desnity and align...,"degen chnges , mild scoliosis convexity to left"
3,045ea333-6d30-4edf-9d33-8ed3feb0772d,MRI,Lumbar spine,MRI LUMBAR,50-54,male,FINDINGS:\nVERTEBRAE: Normal vertebral body he...,Straightening of the normal lumbar lordosis is...
4,06ef946f-5b3b-4b7c-a64f-58c184dca4e6,XRAY,Shoulder,XR L SHOULDER 2V,65-69,male,FINDINGS:\nBONES: No fracture or focal lesion....,Bones show no acute fracture or dislocation. G...


In [6]:
test['template_content'][0].split("\n")

['FINDINGS:',
 'LIVER: Normal in size and attenuation. No focal hepatic lesion.',
 'GALLBLADDER AND BILIARY TREE: Normal. No biliary ductal dilatation.',
 'PANCREAS: Normal in size and contour.',
 'SPLEEN: Normal.',
 'ADRENAL GLANDS: Normal in size and morphology.',
 'KIDNEYS AND URETERS: Normal appearance without renal calculi or hydronephrosis.',
 'URINARY BLADDER: Normal. No calculus or mass lesion.',
 'REPRODUCTIVE: The uterus and ovaries/prostate are normal in appearance.',
 'MAJOR VESSELS: The abdominal aorta is normal in caliber.',
 'PERITONEUM: No ascites. No pneumoperitoneum.',
 'ABDOMINAL WALL: The abdominal wall is unremarkable. No hernia.',
 'LYMPH NODES: No significant mesenteric or retroperitoneal lymphadenopathy.',
 'STOMACH AND BOWEL: No evidence of bowel obstruction.',
 'BONES: No acute osseous abnormality.',
 '',
 'IMPRESSION:',
 'No acute intra-abdominal abnormality identified.']

In [7]:
test['dictation'][0].split("\n")

['Surgically absent gallbladder. Surgical clips are seen in the gallbladder fossa. ',
 'small splenic calcification',
 'pelvic phleboltihs',
 'appendix is surgically absent, with surgical suture at the cecum.',
 'Colonic diverticulosis is present. There is focal moderate pericolonic fat stranding adjacent to the sigmoid colon, best seen on series 201 images 180 to 185, consistent with acute diverticulitis. No adjacent fluid collection or abscess is identified. No extraluminal free air is seen.',
 'no bowel obstruction',
 'Enteric contrast is seen from stomach to distal small']

In [8]:
train = pd.read_csv("./radiology-reporting-harness/train.csv")
train.head()

,case_id,modality,body_part,study_description,patient_age_band,patient_sex,template_content,dictation,report
0,2c88d015-b359-4e2c-9c62-61a23ce9d1c3,XRAY,Hip,XR RT HIP 2V,90+,female,FINDINGS:\nBONES: No acute fracture or focal o...,No acute fracture or dislocation. Mild right h...,FINDINGS:\nBONES: No acute fracture. Mild dege...
1,218fc4a5-c7e9-44a7-9aea-54e56bb22663,XRAY,Chest,XR CXR 2V,65-69,male,FINDINGS:\nLUNGS: Lungs are clear. No focal ai...,"no effusuon, infiltrates\nmild thoracic spondy...",FINDINGS:\nLUNGS: Lungs are clear. No focal ai...
2,2066ed76-9f41-4ac8-95f6-74ec1e4cef8a,MRI,Shoulder,MRI RT SHOULDER WO,45-49,male,FINDINGS:\nTENDONS:\nSUPRASPINATUS: The tendon...,MRI RIGHT SHOULDER WITHOUT CONTRAST Right shou...,FINDINGS:\nTENDONS:\nSUPRASPINATUS: There is m...
3,71210e04-aa16-4a47-8c20-fe58494260dc,MRI,Head,MRI Brain^SUBTLE BRAIN,70-74,female,FINDINGS:\nBRAIN: No restricted diffusion to i...,"mild atrophy , mild leuko",FINDINGS:\nBRAIN: There is mild atrophy. There...
4,656574f1-ffea-49f7-a0c8-953a5cd40bfc,MRI,Pelvis,MRI PELVIS,45-49,male,FINDINGS:\nBOWEL: The visualized loops of smal...,The examination is suboptimal due to multiple ...,FINDINGS:\nThe examination is suboptimal due t...


In [9]:
train['template_content'][0].split("\n")

['FINDINGS:',
 'BONES: No acute fracture or focal osseous lesion.',
 'JOINTS: No dislocation. The joint spaces are normal.',
 'SOFT TISSUES: The soft tissues are unremarkable.',
 '',
 'IMPRESSION:',
 'No acute osseous abnormality.']

In [10]:
train['dictation'][0].split("\n")

['No acute fracture or dislocation. Mild right hip osteoarthrosis with mild superolateral joint space narrowing and small marginal acetabular and femoral head osteophytes. Mild degenerative changes of the bilateral sacroiliac joints Mild degenerative changes of the visualized lower lumbar spine. Pelvic ring is intact.']

OLLAMA CLIENT LLM AND EXECUTION

In [11]:
def get_prompt(inputs, version="v2"):
    with open("config.yaml", "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)

    prompt_file = config["prompt"][version]["name"]

    with open(f'./prompts/{prompt_file}', "r", encoding="utf-8") as f:
        prompt_text = f.read()

    prompt = prompt_text.format(
        modality=inputs["modality"],
        body_part=inputs["body_part"],
        study_description=inputs["study_description"],
        patient_age_band=inputs["patient_age_band"],
        patient_sex=inputs["patient_sex"],
        template_content=inputs["template_content"],
        dictation=inputs["dictation"],
        few_shot=few_shot
    )
    
    return prompt

def get_model(version="m4"):
    with open("config.yaml", "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    return {"model": config["model"][version]["name"], "base": config["model"][version]["base"]}

# prompt = get_prompt({
#     "modality": test['modality'][0],
#     "body_part": test['body_part'][0],
#     "study_description": test['study_description'][0],
#     "patient_age_band": test['patient_age_band'][0],
#     "patient_sex": test['patient_sex'][0],
#     "template_content": test['template_content'][0],
#     "dictation": test['dictation'][0]
# })

# response = client.generate(
#         model="deepseek-r1:1.5b",
#         prompt=prompt,
#         stream=False
#     )

# response["response"]

In [16]:
import re
from litellm import completion

few_shot_input = train.iloc[0].drop("report").to_string()
few_shot_output = train.iloc[0]["report"]
few_shot = f"Input:\n {few_shot_input}\nOutput:\n {few_shot_output}"


def clean_report(text):
    text = re.split(r"\bFINAL OUTPUT FORMAT\b", text, maxsplit=1, flags=re.IGNORECASE)[0]
    text = re.sub(r"```(?:plaintext|text|json)?", "", text, flags=re.IGNORECASE)
    text = text.replace("```", "")
    text = re.sub(r"\*{1,2}(FINDINGS|IMPRESSION):\*{1,2}", r"\1:", text, flags=re.IGNORECASE)
    text = re.sub(r"(?im)^\s*\[.*?\]\s*$", "", text)
    text = re.sub(r"(?im)^\s*(?:EXPLANATION|NOTE):.*$", "", text)

    findings_match = re.search(r"\bFINDINGS:\s*", text, flags=re.IGNORECASE)
    if not findings_match:
        return text.strip()

    impression_match = re.search(
        r"\bIMPRESSION:\s*", text[findings_match.end():], flags=re.IGNORECASE
    )
    if not impression_match:
        return text[findings_match.start():].strip()

    impression_start = findings_match.end() + impression_match.start()
    impression_content_start = findings_match.end() + impression_match.end()
    findings = text[findings_match.end():impression_start].strip()
    impression = text[impression_content_start:]
    impression = re.split(
        r"\b(?:FINDINGS|IMPRESSION):\s*|\b(?:EXPLANATION|NOTE):\s*|\b(?:Certainly|Feel free)\b",
        impression,
        maxsplit=1,
        flags=re.IGNORECASE,
    )[0].strip().rstrip("` ")

    # Remove the entire unwanted block, including blank, None, or populated content.
    findings = re.sub(
        r"(?ims)^\s*OTHER\s+FINDINGS\s*:\s*.*?(?=^\s*[A-Z][A-Z0-9 /&()_-]*\s*:|\Z)",
        "",
        findings,
    )
    findings = re.sub(r"\n{3,}", "\n\n", findings).strip()
    return f"FINDINGS:\n{findings}\n\nIMPRESSION:\n{impression}"


def callLLM(inputs):
    prompt = get_prompt(inputs)
    model_info = get_model("m5")
    response = completion(
        model=model_info["model"],
        base=model_info["base"],
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=1024,
        stream=False,
    )
    return clean_report(response.choices[0].message.content)

In [17]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

submission_path = Path("submission.csv")
append_to_submission = False
submission_initialized = append_to_submission and submission_path.exists()
max_workers = 2

def generate_result(row_number):
    row = test.iloc[row_number]
    res = callLLM(row.drop("case_id").to_dict())
    return {"case_id": row["case_id"], "report": res}

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    for batch_start in range(0, len(test), 5):
        batch_end = min(batch_start + 5, len(test))
        print(f"\nBatch {batch_start // 5}\n")

        results = executor.map(generate_result, range(batch_start, batch_end))
        for iteration, result in enumerate(results):
            print(f"\niter {iteration}\n")
            pd.DataFrame([result]).to_csv(
                submission_path,
                mode="a" if submission_initialized else "w",
                header=not submission_initialized,
                index=False,
            )
            submission_initialized = True
            print(result["report"])


Batch 0


iter 0

FINDINGS:
LIVER: Normal in size and attenuation. No focal hepatic lesion.
GALLBLADDER AND BILIARY TREE: The gallbladder is surgically absent with surgical clips in the gallbladder fossa.
PANCREAS: Normal in size and contour.
SPLEEN: Small splenic calcification is present.
ADRENAL GLANDS: Normal in size and morphology.
KIDNEYS AND URETERS: Normal appearance without renal calculi or hydronephrosis.
URINARY BLADDER: Normal. No calculus or mass lesion.
REPRODUCTIVE: The uterus and ovaries/prostate are normal in appearance.
MAJOR VESSELS: The abdominal aorta is normal in caliber.
PERITONEUM: No ascites. No pneumoperitoneum.
ABDOMINAL WALL: The abdominal wall is unremarkable. No hernia.
LYMPH NODES: No significant mesenteric or retroperitoneal lymphadenopathy.
STOMACH AND BOWEL: Colonic diverticulosis is present with focal moderate pericolonic fat stranding adjacent to the sigmoid colon, consistent with acute diverticulitis. No adjacent fluid collection or abscess is ident

In [14]:
print(len(test), len(pd.read_csv(submission_path)))

132 132
